In [2]:
import pandas as pd

# Aggregate Monthly Enrollment Data by Group (2014-2019)

In [2]:
enrollment_by_rg = pd.read_excel("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/medicaid_&_chip_enrollement/monthly-enrollment-by-risk-group.xlsx", sheet_name='Caseload by RG', skiprows=2)

In [3]:
 # the last few rows are blank or contain notes, so we trim to just the data
enrollment_by_rg = enrollment_by_rg[0:138]

# standardize the column names to be lowercase, with underscores instead of spaces, and no special characters. This is necessary for loading into Snowflake.
enrollment_by_rg.columns = (enrollment_by_rg.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('*', '', regex=False)
    .str.replace('&', 'and', regex=False)
    .str.replace('-', '_', regex=False)
    .str.replace('\u2019', '', regex=False)  # curly apostrophe
    .str.replace("'", '', regex=False)        # straight apostrophe fallback
    .str.replace('.', '_', regex=False)       # handles the .1 suffix
)

# rename the columns to be more descriptive and clear about the risk groups.
enrollment_by_rg = enrollment_by_rg.rename(columns={
    'childrens_medicaid':   'childrens_medicaid_risk_group',
    'childrens_medicaid_1': 'childrens_medicaid_chip_group',
    'total':                'childrens_and_chip_total'
})

# add a timestamp column to track when the data was loaded into Snowflake
enrollment_by_rg['loaded_at'] = pd.Timestamp.now()

# Aggregate CHIP Enrollment Data (2014-2019)

In [ ]:
chip_enrollment = pd.read_excel("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/medicaid_&_chip_enrollement/chip-enrollment-detail.xlsx", sheet_name='CHIP Regular Caseload', skiprows=1)

# the last few rows are blank or contain notes, so we trim to just the data
chip_enrollment = chip_enrollment[0:138]

# standardize the column names to be lowercase, with underscores instead of spaces, and no special characters. This is necessary for loading into Snowflake.
chip_enrollment.columns = (chip_enrollment.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('*', '', regex=False)
    .str.replace('&', 'and', regex=False)
    .str.replace('-', '_', regex=False)
    .str.replace('\u2019', '', regex=False)  # curly apostrophe
    .str.replace("'", '', regex=False)        # straight apostrophe fallback
    .str.replace('.', '_', regex=False)       # handles the .1 suffix
)

# add a timestamp column to track when the data was loaded into Snowflake
chip_enrollment['loaded_at'] = pd.Timestamp.now()

# Healthy Texas Women Enrollment (2014-2019)

In [ ]:
hwt_enrollment = pd.read_excel("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/medicaid_&_chip_enrollement/healthy-texas-women-enrollment.xlsx", sheet_name='Summary')

# the last few rows are blank or contain notes, so we trim to just the data
hwt_enrollment = hwt_enrollment[0:138]

# standardize the column names to be lowercase, with underscores instead of spaces, and no special characters. This is necessary for loading into Snowflake.
hwt_enrollment.columns = (hwt_enrollment.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('*', '', regex=False)
    .str.replace('&', 'and', regex=False)
    .str.replace('-', '_', regex=False)
    .str.replace('\u2019', '', regex=False)  # curly apostrophe
    .str.replace("'", '', regex=False)        # straight apostrophe fallback
    .str.replace('.', '_', regex=False)       # handles the .1 suffix
)

# rename the columns to be more descriptive and clear about the risk groups.
hwt_enrollment = hwt_enrollment.rename(columns={
    'healthy_texas_women_caseload':   'month',
    'unnamed:_1': 'caseload',
})

# add a timestamp column to track when the data was loaded into Snowflake
hwt_enrollment['loaded_at'] = pd.Timestamp.now()

In [30]:
print(hwt_enrollment.head())

                 month       caseload                  loaded_at
0  2026-02-01 00:00:00  357189.038319 2026-05-15 01:12:16.945961
1  2026-01-01 00:00:00  359137.461471 2026-05-15 01:12:16.945961
2  2025-12-01 00:00:00  365103.390303 2026-05-15 01:12:16.945961
3  2025-11-01 00:00:00  369671.509855 2026-05-15 01:12:16.945961
4  2025-10-01 00:00:00  377313.683691 2026-05-15 01:12:16.945961


In [32]:
print(len(enrollment_by_rg))       # should be 138
print(len(chip_enrollment))  # should be 138
print(len(hwt_enrollment))   # should be 138

138
138
138


# Medicaid Timeliness Data 

In [12]:
apps = pd.read_excel("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/timeliness/timeliness-medicaid-jan-2025.xlsx", sheet_name='Medicaid', skiprows=3)

redets = pd.read_excel("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/timeliness/timeliness-medicaid-jan-2025.xlsx", sheet_name='Medicaid', skiprows=25)

In [13]:
apps = apps[1:19]  # trim first empty row and total row

redets = redets[1:19]  # trim first empty row and total row


In [14]:
print(len(apps))  
print(len(redets))

18
18


In [15]:
print(apps.head(3))
print(redets.head(3))

  Region Disposed Timely   Percent
1     01    22469  18434  0.820419
2  02/09    15149  11720  0.773648
3     03    48393  40082   0.82826
  Region  Disposed   Timely   Percent
1     01    9415.0   9384.0  0.996707
2  02/09    7012.0   7000.0  0.998289
3     03   22668.0  22641.0  0.998809


In [18]:
from pathlib import Path
import re

timeliness_folder = Path("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/timeliness/")

all_timeliness = []

for file in sorted(timeliness_folder.glob("*.xlsx")):
    # extract report_month from filename
    match = re.search(r'medicaid-(\w+-\d{4})\.xlsx', file.name)
    if not match:
        print(f"Skipping {file.name} - could not parse month")
        continue
    report_month = pd.to_datetime(match.group(1), format='mixed')
    
    # read applications table
    apps = pd.read_excel(file, sheet_name='Medicaid', skiprows=3)
    apps = apps[1:19]
    apps['record_type'] = 'applications'
    
    # read redeterminations table
    redets = pd.read_excel(file, sheet_name='Medicaid', skiprows=25)
    redets = redets[1:19]
    redets['record_type'] = 'redeterminations'
    
    # union both tables
    combined = pd.concat([apps, redets], ignore_index=True)
    
    # add report_month
    combined['report_month'] = report_month
    
    all_timeliness.append(combined)
    print(f"Processed {file.name}: {len(combined)} rows")

# concatenate all months
timeliness = pd.concat(all_timeliness, ignore_index=True)
print(f"\nTotal rows: {len(timeliness)}")

Processed timeliness-medicaid-april-2024.xlsx: 36 rows
Processed timeliness-medicaid-april-2025.xlsx: 36 rows
Processed timeliness-medicaid-aug-2024.xlsx: 36 rows
Processed timeliness-medicaid-aug-2025.xlsx: 36 rows
Processed timeliness-medicaid-dec-2024.xlsx: 36 rows
Processed timeliness-medicaid-dec-2025.xlsx: 36 rows
Processed timeliness-medicaid-feb-2024.xlsx: 36 rows
Processed timeliness-medicaid-feb-2025.xlsx: 36 rows
Processed timeliness-medicaid-jan-2024.xlsx: 36 rows
Processed timeliness-medicaid-jan-2025.xlsx: 36 rows
Processed timeliness-medicaid-july-2024.xlsx: 36 rows
Processed timeliness-medicaid-july-2025.xlsx: 36 rows
Processed timeliness-medicaid-june-2024.xlsx: 36 rows
Processed timeliness-medicaid-june-2025.xlsx: 36 rows
Processed timeliness-medicaid-march-2024.xlsx: 36 rows
Processed timeliness-medicaid-march-2025.xlsx: 36 rows
Processed timeliness-medicaid-may-2024.xlsx: 36 rows
Processed timeliness-medicaid-may-2025.xlsx: 36 rows
Processed timeliness-medicaid-nov-

In [19]:
print(timeliness.columns.tolist())
print(timeliness.head(3))

['Region', 'Disposed', 'Timely', 'Percent', 'record_type', 'report_month']
  Region Disposed Timely   Percent   record_type report_month
0     01    18192   6260  0.344107  applications   2024-04-01
1  02/09    19480  10561  0.542146  applications   2024-04-01
2     03    65717  29251  0.445106  applications   2024-04-01


In [20]:
print(timeliness['record_type'].value_counts())

record_type
applications        432
redeterminations    432
Name: count, dtype: int64


In [22]:
timeliness.columns = (timeliness.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
)

timeliness['loaded_at'] = pd.Timestamp.now()

geographic_regions = ['01', '02/09', '03', '04', '05', '06', '07', '08', '10', '11']
timeliness['is_geographic_region'] = timeliness['region'].isin(geographic_regions)

In [23]:
timeliness['is_geographic_region'].value_counts()

is_geographic_region
True     480
False    384
Name: count, dtype: int64

# Medicaid Enrollment by County 

In [30]:
county_enrollment = pd.read_excel('/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/county/medicaid-enrollment-by-county-final-april-2024.xlsx', sheet_name='Summary', skiprows=2)

county_enrollment = county_enrollment[0:255] # trim to just the data rows, excluding notes at the end

In [32]:
county_enrollment

,HHSC County Code,County,Medicaid Caseload,Aged & Medicare Related,Disability-Related,Parents*,Pregnant Women,Breast and Cervical Cancer,Children's Medicaid,Medicaid Clients Under 21**,Medicaid Clients 21 and Older
0,1,Anderson,8272.0,867.0,828.0,289.0,615.0,4.0,5669.0,5986.0,2286.0
1,2,Andrews,2398.0,173.0,126.0,88.0,208.0,1.0,1802.0,1881.0,517.0
2,3,Angelina,15684.0,1540.0,1820.0,495.0,1084.0,15.0,10730.0,11508.0,4176.0
3,4,Aransas,3529.0,370.0,372.0,142.0,231.0,11.0,2403.0,2491.0,1038.0
4,5,Archer,697.0,65.0,74.0,18.0,54.0,1.0,485.0,516.0,181.0
...,...,...,...,...,...,...,...,...,...,...,...
250,251,Yoakum,1141.0,89.0,51.0,40.0,85.0,4.0,872.0,898.0,243.0
251,252,Young,2589.0,285.0,256.0,77.0,183.0,3.0,1785.0,1880.0,709.0
252,253,Zapata,3885.0,417.0,284.0,98.0,189.0,1.0,2896.0,3042.0,843.0
253,254,Zavala,3048.0,355.0,294.0,111.0,183.0,6.0,2099.0,2226.0,822.0


In [33]:
import pandas as pd

f = "/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/county/medicaid-enrollment-by-county-final-jan-2024.xlsx"

xl = pd.ExcelFile(f)
print("Sheets:", xl.sheet_names)

df = pd.read_excel(f, sheet_name=xl.sheet_names[0], nrows=10)
print(df)

Sheets: ['Summary']
  Final Count - Medicaid Enrollment by County - January 2024 Unnamed: 1  \
0                                                NaN                NaN   
1                                   HHSC County Code             County   
2                                                  1           Anderson   
3                                                  2            Andrews   
4                                                  3           Angelina   
5                                                  4            Aransas   
6                                                  5             Archer   
7                                                  6          Armstrong   
8                                                  7           Atascosa   
9                                                  8             Austin   

          Unnamed: 2               Unnamed: 3          Unnamed: 4 Unnamed: 5  \
0                NaN   Caseload by Risk Group                 NaN        N

In [34]:
import pandas as pd

f = "/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/county/medicaid-enrollment-by-county-final-jan-2024.xlsx"

df = pd.read_excel(f, sheet_name='Summary', skiprows=2)
print(df.tail(5))
print("Total rows:", len(df))


                                      HHSC County Code County  \
260  *Parents includes TANF Adults and Medically Needy    NaN   
261  **Medicaid Clients Under 21 includes all full ...    NaN   
262                       and Cervical Cancer Clients)    NaN   
263                               Source: E8mth Tables    NaN   
264                                   HHSC Forecasting    NaN   

     Medicaid Caseload  Aged & Medicare Related  Disability-Related  Parents*  \
260                NaN                      NaN                 NaN       NaN   
261                NaN                      NaN                 NaN       NaN   
262                NaN                      NaN                 NaN       NaN   
263                NaN                      NaN                 NaN       NaN   
264                NaN                      NaN                 NaN       NaN   

     Pregnant Women  Breast and Cervical Cancer  Children's Medicaid  \
260             NaN                         NaN   

# MCO Enrollment by SDA

In [35]:
mco = pd.read_excel("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/medicaid_&_chip_enrollement/mco-enrollment-by-sda-final-sfy25.xlsx", sheet_name="Caseload_source", skiprows=1)

In [ ]:
mco = mco[0:203] # trim to just the data rows, excluding notes at the end

In [40]:
mco

,Unnamed: 0,Dallas,Tarrant,Harris,Nueces,Bexar,Travis,El Paso,Lubbock,Hildago,Jefferson,MRSA Central,MRSA North,MRSA West,EPO*,Total
0,Wellpoint / Amerigroup,231809.083333,110900.500000,7.289092e+04,6453.083333,11006.583333,0.000000,1375.500000,15156.666667,0.000000,17207.500000,16226.083333,62856.833333,42837.000000,0.00,5.887198e+05
1,CHIP,14979.750000,6589.500000,3.215417e+03,0.000000,793.416667,0.000000,0.000000,0.000000,0.000000,267.000000,0.000000,0.000000,0.000000,0.00,2.584508e+04
2,Regular,13099.250000,5747.833333,2.143083e+03,NaN,510.250000,NaN,NaN,NaN,NaN,183.750000,NaN,NaN,NaN,NaN,2.168417e+04
3,Perinatal,1880.500000,841.666667,1.072333e+03,NaN,283.166667,NaN,NaN,NaN,NaN,83.250000,NaN,NaN,NaN,NaN,4.160917e+03
4,MEDICAID,216829.333333,104311.000000,6.967550e+04,6453.083333,10213.166667,0.000000,1375.500000,15156.666667,0.000000,16940.500000,16226.083333,62856.833333,42837.000000,0.00,5.628747e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,Total CHIP****,32390.250000,22244.250000,5.155983e+04,4098.000000,15584.666667,12531.250000,6822.500000,4258.333333,0.000000,4415.833333,0.000000,0.000000,0.000000,42990.25,1.968952e+05
212,Total Caseload,560158.666667,400142.250000,1.042243e+06,123366.250000,382125.250000,220897.333333,154447.583333,105367.500000,440208.000000,131026.666667,197273.333333,248747.250000,218263.500000,47186.00,4.271452e+06
213,Percent Managed Care,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
214,Medicaid,0.939705,0.932549,9.456283e-01,0.952469,0.943171,0.927014,0.948312,0.935540,0.963438,0.947463,0.942474,0.942639,0.938193,0.00,9.427961e-01


In [41]:
import pandas as pd

f = "/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/medicaid_&_chip_enrollement/mco-enrollment-by-sda-final-sfy25.xlsx"

df = pd.read_excel(f, sheet_name=0, header=None, nrows=25)
print(df.to_string())

                        0              1              2             3             4              5              6             7             8              9            10            11             12            13       14             15
0   State Fiscal Year 2025            NaN            NaN           NaN           NaN            NaN            NaN           NaN           NaN            NaN          NaN           NaN            NaN           NaN      NaN            NaN
1                      NaN         Dallas        Tarrant        Harris        Nueces          Bexar         Travis       El Paso       Lubbock        Hildago    Jefferson  MRSA Central     MRSA North     MRSA West     EPO*          Total
2   Wellpoint / Amerigroup  231809.083333       110900.5  72890.916667   6453.083333   11006.583333              0        1375.5  15156.666667              0      17207.5  16226.083333   62856.833333         42837        0      588719.75
3                     CHIP       14979.75       

In [43]:
df = pd.read_excel("/Users/jameslavin/Documents/dev/hhsc_data_architect_project/data/raw/medicaid_&_chip_enrollement/mco-enrollment-by-sda-final-sfy25.xlsx", sheet_name=0, header=None)

# row 1 contains the SDA/column names; drop the last column (Total)
sda_columns = df.iloc[1, 1:-1].tolist()

# MCO blocks start at row 2, each block is exactly 10 rows
# rows 0-201 are data; row 202 onward is statewide totals — drop those
MCO_BLOCK_SIZE = 10
DATA_START_ROW = 2
DATA_END_ROW   = 202  # exclusive

# sub-program labels in fixed order within each MCO block
# row 0 = MCO total, row 1 = CHIP total, row 2 = Regular, row 3 = Perinatal
# row 4 = MEDICAID total, row 5 = STAR, row 6 = STAR+Plus, row 7 = Dual Demo
# row 8 = STAR Health, row 9 = STAR Kids
PROGRAM_MAP = {
    0: ('TOTAL',    'TOTAL'),
    1: ('CHIP',     'TOTAL'),
    2: ('CHIP',     'Regular'),
    3: ('CHIP',     'Perinatal'),
    4: ('MEDICAID', 'TOTAL'),
    5: ('MEDICAID', 'STAR'),
    6: ('MEDICAID', 'STAR+Plus'),
    7: ('MEDICAID', 'Dual Demo'),
    8: ('MEDICAID', 'STAR Health'),
    9: ('MEDICAID', 'STAR Kids'),
}

records = []

for block_start in range(DATA_START_ROW, DATA_END_ROW, MCO_BLOCK_SIZE):
    mco_name = df.iloc[block_start, 0]

    for offset, (program, sub_program) in PROGRAM_MAP.items():
        row = df.iloc[block_start + offset, 1:-1]  # drop first col (label) and last col (total)

        for sda, value in zip(sda_columns, row):
            # NaN means the MCO does not serve that SDA — skip entirely
            if pd.isna(value):
                continue

            records.append({
                'mco_name':        mco_name,
                'program':         program,
                'sub_program':     sub_program,
                'sda':             sda,
                'enrollment':      value,
                'enrollment_type': 'sfy_monthly_average',
                'fiscal_year':     2025,
                'source_file':     'mco-enrollment-by-sda-final-sfy25.xlsx',
                'loaded_at':       pd.Timestamp.now()
            })

mco_sda = pd.DataFrame(records)

# clean the mco_name column — one MCO has a newline character in its name
mco_sda['mco_name'] = mco_sda['mco_name'].str.replace('\n', ' ', regex=False).str.strip()

print(f"Total rows: {len(mco_sda)}")
print(f"MCOs: {sorted(mco_sda['mco_name'].unique())}")
print(f"SDAs: {sorted(mco_sda['sda'].unique())}")
print(mco_sda.head(10))

Total rows: 1004
MCOs: ['Aetna', 'Blue Cross and Blue Shield of Texas', 'CHRISTUS', "Children's Medical Center", 'Cigna/Texas HealthSpring', 'Community First', 'Community Health Choice', "Cook Children's", 'Dell Children', 'Driscoll', 'El Paso First', 'FirstCare', 'Molina Healthcare of Texas', 'Parkland Community', 'Scott & White', 'Sendero', 'Superior', "Texas Children's", 'UnitedHealthcare/ Evercare of Texas', 'Wellpoint / Amerigroup']
SDAs: ['Bexar', 'Dallas', 'EPO*', 'El Paso', 'Harris', 'Hildago', 'Jefferson', 'Lubbock', 'MRSA Central', 'MRSA North', 'MRSA West', 'Nueces', 'Tarrant', 'Travis']
                 mco_name program sub_program        sda     enrollment  \
0  Wellpoint / Amerigroup   TOTAL       TOTAL     Dallas  231809.083333   
1  Wellpoint / Amerigroup   TOTAL       TOTAL    Tarrant  110900.500000   
2  Wellpoint / Amerigroup   TOTAL       TOTAL     Harris   72890.916667   
3  Wellpoint / Amerigroup   TOTAL       TOTAL     Nueces    6453.083333   
4  Wellpoint / Amer

In [45]:
mco_sda

,mco_name,program,sub_program,sda,enrollment,enrollment_type,fiscal_year,source_file,loaded_at
0,Wellpoint / Amerigroup,TOTAL,TOTAL,Dallas,231809.083333,sfy_monthly_average,2025,mco-enrollment-by-sda-final-sfy25.xlsx,2026-05-17 18:11:54.650759
1,Wellpoint / Amerigroup,TOTAL,TOTAL,Tarrant,110900.500000,sfy_monthly_average,2025,mco-enrollment-by-sda-final-sfy25.xlsx,2026-05-17 18:11:54.650841
2,Wellpoint / Amerigroup,TOTAL,TOTAL,Harris,72890.916667,sfy_monthly_average,2025,mco-enrollment-by-sda-final-sfy25.xlsx,2026-05-17 18:11:54.650843
3,Wellpoint / Amerigroup,TOTAL,TOTAL,Nueces,6453.083333,sfy_monthly_average,2025,mco-enrollment-by-sda-final-sfy25.xlsx,2026-05-17 18:11:54.650845
4,Wellpoint / Amerigroup,TOTAL,TOTAL,Bexar,11006.583333,sfy_monthly_average,2025,mco-enrollment-by-sda-final-sfy25.xlsx,2026-05-17 18:11:54.650846
...,...,...,...,...,...,...,...,...,...
999,Children's Medical Center,MEDICAID,TOTAL,Jefferson,0.000000,sfy_monthly_average,2025,mco-enrollment-by-sda-final-sfy25.xlsx,2026-05-17 18:11:54.657722
1000,Children's Medical Center,MEDICAID,TOTAL,MRSA Central,0.000000,sfy_monthly_average,2025,mco-enrollment-by-sda-final-sfy25.xlsx,2026-05-17 18:11:54.657723
1001,Children's Medical Center,MEDICAID,TOTAL,MRSA North,0.000000,sfy_monthly_average,2025,mco-enrollment-by-sda-final-sfy25.xlsx,2026-05-17 18:11:54.657724
1002,Children's Medical Center,MEDICAID,TOTAL,MRSA West,0.000000,sfy_monthly_average,2025,mco-enrollment-by-sda-final-sfy25.xlsx,2026-05-17 18:11:54.657725
